<a href="https://colab.research.google.com/github/Md-Golam-Raiyhan/INSE-6320-Project/blob/main/risk_assesment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd

# ==========================================
# Load dataset
# ==========================================
file_path = "metro_rail_project_dataset.csv"
df = pd.read_csv(file_path)

# ==========================================
# Define risk variables and impact scores
# ==========================================
risk_info = {
    "material_price_index": {
        "Risk": "Material Price Volatility",
        "Impact_Score": 3
    },
    "labor_availability_index": {
        "Risk": "Labor Availability",
        "Impact_Score": 2
    },
    "weather_delay_days": {
        "Risk": "Weather Delay",
        "Impact_Score": 2
    },
    "permit_delay_days": {
        "Risk": "Permit Delay",
        "Impact_Score": 3
    },
    "equipment_downtime_hours": {
        "Risk": "Equipment Downtime",
        "Impact_Score": 2
    },
    "inflation_rate_annual": {
        "Risk": "Inflation Rate",
        "Impact_Score": 3
    },
    "change_orders": {
        "Risk": "Change Orders",
        "Impact_Score": 3
    },
    "ground_risk_score": {
        "Risk": "Ground Risk",
        "Impact_Score": 3
    },
    "contractor_experience_score": {
        "Risk": "Contractor Experience",
        "Impact_Score": 2
    }
}

# ==========================================
# Compute probability, impact, risk score
# ==========================================
rows = []

for var, info in risk_info.items():
    mean_val = df[var].mean()
    std_val = df[var].std()

    # high-risk threshold
    threshold = mean_val + std_val

    # for labor availability and contractor experience,
    # lower values are more risky
    if var in ["labor_availability_index", "contractor_experience_score"]:
        probability = (df[var] < (mean_val - std_val)).mean()
    else:
        probability = (df[var] > threshold).mean()

    impact = info["Impact_Score"]
    risk_score = probability * impact

    rows.append({
        "Dataset_Variable": var,
        "Risk": info["Risk"],
        "Probability": round(probability, 4),
        "Impact_Score": impact,
        "Risk_Score": round(risk_score, 4)
    })

risk_df = pd.DataFrame(rows)

# ==========================================
# Assign priority based on risk score ranking
# ==========================================
risk_df = risk_df.sort_values("Risk_Score", ascending=False).reset_index(drop=True)
risk_df["Priority"] = range(1, len(risk_df) + 1)

# ==========================================
# Save to single CSV
# ==========================================
output_file = "risk_assessment_summary.csv"
risk_df.to_csv(output_file, index=False)

print(risk_df)
print(f"\nSaved file: {output_file}")

              Dataset_Variable                       Risk  Probability  \
0            ground_risk_score                Ground Risk       0.2091   
1                change_orders              Change Orders       0.1818   
2         material_price_index  Material Price Volatility       0.1545   
3        inflation_rate_annual             Inflation Rate       0.1545   
4            permit_delay_days               Permit Delay       0.1455   
5  contractor_experience_score      Contractor Experience       0.2045   
6           weather_delay_days              Weather Delay       0.2000   
7     equipment_downtime_hours         Equipment Downtime       0.1773   
8     labor_availability_index         Labor Availability       0.1636   

   Impact_Score  Risk_Score  Priority  
0             3      0.6273         1  
1             3      0.5455         2  
2             3      0.4636         3  
3             3      0.4636         4  
4             3      0.4364         5  
5             2    

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ==========================================
# Load risk assessment summary
# ==========================================
file_path = "risk_assessment_summary.csv"
df = pd.read_csv(file_path)

# output folder
output_dir = Path("clean_risk_matrix_outputs")
output_dir.mkdir(exist_ok=True)

# ==========================================
# Create short numeric labels for plotting
# ==========================================
df = df.sort_values("Priority").reset_index(drop=True)
df["Label"] = range(1, len(df) + 1)

# ==========================================
# Create cleaner probability bands
# ==========================================
# You can adjust these thresholds if needed
def prob_band(p):
    if p < 0.17:
        return 1   # Low
    elif p < 0.19:
        return 2   # Medium
    else:
        return 3   # High

df["Probability_Band"] = df["Probability"].apply(prob_band)

# ==========================================
# Plot clean qualitative risk matrix
# ==========================================
plt.figure(figsize=(8, 6))

# background colored zones
for x in [0.5, 1.5, 2.5]:
    for y in [0.5, 1.5, 2.5]:
        # choose color by severity
        severity = x + y
        if severity <= 3:
            color = "#d9ead3"   # green
        elif severity <= 4:
            color = "#fff2cc"   # yellow
        else:
            color = "#f4cccc"   # red
        rect = plt.Rectangle((x, y), 1, 1, facecolor=color, edgecolor="gray", alpha=0.8)
        plt.gca().add_patch(rect)

# scatter points
plt.scatter(
    df["Probability_Band"],
    df["Impact_Score"],
    s=180,
    edgecolor="black"
)

# add numeric labels inside markers
for _, row in df.iterrows():
    plt.text(
        row["Probability_Band"],
        row["Impact_Score"],
        str(row["Label"]),
        ha="center",
        va="center",
        fontsize=9,
        fontweight="bold"
    )

# axis formatting
plt.xlim(0.5, 3.5)
plt.ylim(0.5, 3.5)
plt.xticks([1, 2, 3], ["Low", "Medium", "High"])
plt.yticks([1, 2, 3], ["Low", "Medium", "High"])
plt.xlabel("Probability")
plt.ylabel("Impact")
plt.title("Risk Matrix")
plt.grid(False)
plt.tight_layout()

# save figure
plt.savefig(output_dir / "clean_risk_matrix.png", dpi=300, bbox_inches="tight")
plt.close()

# ==========================================
# Save legend table for report
# ==========================================
legend_table = df[["Label", "Risk", "Probability", "Impact_Score", "Risk_Score", "Priority"]]
legend_table.to_csv(output_dir / "risk_matrix_legend_table.csv", index=False)

print("Saved:")
print(output_dir / "clean_risk_matrix.png")
print(output_dir / "risk_matrix_legend_table.csv")
print("\nLegend Table:")
print(legend_table)

Saved:
clean_risk_matrix_outputs/clean_risk_matrix.png
clean_risk_matrix_outputs/risk_matrix_legend_table.csv

Legend Table:
   Label                       Risk  Probability  Impact_Score  Risk_Score  \
0      1                Ground Risk       0.2091             3      0.6273   
1      2              Change Orders       0.1818             3      0.5455   
2      3  Material Price Volatility       0.1545             3      0.4636   
3      4             Inflation Rate       0.1545             3      0.4636   
4      5               Permit Delay       0.1455             3      0.4364   
5      6      Contractor Experience       0.2045             2      0.4091   
6      7              Weather Delay       0.2000             2      0.4000   
7      8         Equipment Downtime       0.1773             2      0.3545   
8      9         Labor Availability       0.1636             2      0.3273   

   Priority  
0         1  
1         2  
2         3  
3         4  
4         5  
5         